# Lecture: LLM Anatomy — Counting Parameters, Memory, and Compute

Read this before opening notebook.ipynb. Each vocabulary term below gets its own section
(marked §) with a picture or worked numbers — skim the index, study the ones that feel fuzzy.

## Learning goals

- Break a Transformer into its parameter chunks: embeddings, attention, MLP (§4).
- Compute KV-cache bytes from layers, heads, context, and dtype (§5–§6).
- Explain why long context is taxed by cache bytes and attention flops, not weights (§7–§8).


## The big idea

Think of an LLM as an office tower. **Parameters** are the filing cabinets built into every floor (each attention + MLP **block** holds millions of learned numbers; the **embeddings** are the lobby directory mapping words to vectors; width is the **hidden size H**, floors are the **layers L**, parallel sub-votes are the **attention heads**). The **KV-cache** is the scratch paper the tower keeps while answering: for every token already read, each floor saved two sticky notes (**key** and **value**), so a longer **context** means a taller paper stack — double the context, double the paper, regardless of cabinet size. Slimming tricks help: **GQA** shares sticky notes across query heads via fewer **kv-heads**, smaller **dtypes** use fewer **bytes** per number, and **tied embeddings** share one directory between entrance and exit. Each new token's cost (**FLOPs per token**) is roughly "reread all cabinets once" (2 × params) plus "compare against the whole paper stack" (attention term growing with context). That is why a 24GB GPU fits an 8B model at 4K context but drowns at 128K: the weights didn't change, the paper stack did.


In [ ]:
import numpy as np  # bring in numpy for the memory arithmetic
import matplotlib.pyplot as plt  # bring in pyplot for the weights-vs-cache bars
np.random.seed(12)  # fix the seed (no randomness here, habit for parity)
P = 8.03e9  # 8B-class parameter count (published total)
weights = P * 2 / 1e9  # weight memory in GB at bf16 (2 bytes each)
L, H, nh, nkv = 32, 4096, 32, 8  # 8B-class shape: layers, width, q-heads, kv-heads
kv_dim = H // nh * nkv  # kv width per layer after GQA grouping: 1024
seqs = np.array([4096, 32768, 131072])  # contexts: 4K, 32K, 128K
cache = 2 * L * kv_dim * seqs * 2 / 1e9  # KV-cache GB: 2(K+V) x L x kv_dim x seq x bf16
print('weights (bf16):', round(float(weights), 1), 'GB (flat: never changes with context)')  # expect 16.1
print('cache 4K / 32K / 128K:', np.round(cache, 1), 'GB (grows with context)')  # expect [0.5, 4.3, 17.2]
print('fits 24GB card at 4K:', bool(weights + float(cache[0]) < 24))  # True: room to spare
print('fits 24GB card at 128K:', bool(weights + float(cache[2]) < 24))  # False: the paper wins
fig, ax = plt.subplots(figsize=(7, 4))  # make one panel for the punchline bars
ax.bar(['weights', 'cache 4K', 'cache 32K', 'cache 128K'], [weights, cache[0], cache[1], cache[2]], color=['steelblue', 'lightgreen', 'orange', 'tomato'])  # blue = flat, rest grow
ax.axhline(24, color='red', linestyle='--', label='24GB card')  # the ceiling that 128K breaks
ax.set_ylabel('GB (bf16, batch 1)')  # label the memory axis
ax.set_title('Weights stay flat; the paper stack sinks the GPU at 128K')  # title the takeaway
ax.legend()  # show what the red line means
plt.tight_layout()  # snug the layout
plt.show()  # render the figure


## Vocabulary, one section each

Quick map — jump to whatever feels fuzzy. §4 audits the cabinets, §5–§6 weigh the paper, §7–§8 price the work and the tricks.

| Term | One-liner | Section |
|---|---|---|
| Parameter | One learned number in the model (8B model = 8 billion of them) | §4 |
| Hidden size (H) | Width of the model's vectors (bigger = wider tower floors) | §4 |
| Layers (L) | How many block floors are stacked (costs multiply by L) | §4 |
| Attention heads | Parallel sub-attentions splitting each layer's width | §4 |
| GQA / kv-heads | Sharing key/value heads across query heads to shrink the cache | §5 |
| KV-cache | Saved keys+values per token: 2 × L × kv_dim × seq × bytes | §6 |
| dtype bytes | Bytes per number: bf16 = 2, int8 = 1 (halving halves weights and cache) | §7 |
| FLOPs per token | Math per generated token: ~2P (weights) + attention term growing with context | §7 |
| Context length | Tokens visible at once (4K → 32K = 8× cache tax) | §6 |
| Tied embeddings | Sharing input/output word matrices (saves one V×H matrix) | §8 |


### §4 Parameters, H, L, heads — auditing the cabinets

**In one sentence: parameters are the model's learned numbers, split into the token table (V×H), per-layer attention (Q, K, V, output matrices over hidden size H), per-layer MLP (the biggest chunk), plus small norms and the output head — multiplied by L layers.**

Counting is just matrix bookkeeping, and the lab trusts it because the same recipe reproduces the real TinyGPT exactly: 464,705 params by torch and by hand (match = True). Watch it: our toy audit prints every chunk — token table 6,240 + positions 4,608 + 4 blocks × 111,840 + final norm + head = 464,705 — beside the stacked bars showing Tiny (~0.46B, MLP share 87.0%) against an 8B tower (~8.0B, MLP share 75.0%): the MLP dominates every block, and embeddings matter only in tiny models.


In [ ]:
import numpy as np  # reuse numpy for the parameter arithmetic
import matplotlib.pyplot as plt  # reuse pyplot for the stacked-bar chart
np.random.seed(12)  # fix the seed (no randomness here, habit for parity)
V, d, L, T = 65, 96, 4, 48  # TinyGPT dims: vocab, width, layers, window (lab values)
tok = V * d  # token table: 65 x 96 = 6240
pos = T * d  # position table: 48 x 96 = 4608
ln = 2 * d  # one LayerNorm: weight + bias = 192
qkv = 3 * d * d + 3 * d  # fused QKV matrix + bias: 27936
proj = d * d + d  # output proj + bias: 9312
mlp = (d * 4 * d + 4 * d) + (4 * d * d + d)  # widen-4 MLP pair: 37248 + 36960 = 74208
block = 2 * ln + qkv + proj + mlp  # one block: norms + attention + mlp = 111840
total = tok + pos + L * block + ln + (d * V + V)  # everything + final norm + head = 464705
print('per-block:', block, '(attn', qkv + proj, '+ mlp', mlp, '+ norms', 2 * ln, ')')  # the chunk split
print('TinyGPT total:', total, '(torch agrees: 464705)')  # the audited headline
names = ['TinyGPT-P11', '8B-class LLM']  # two towers to compare
embed = np.array([0.01, 0.50])  # embedding + head params in billions
attn = np.array([0.05, 1.50])  # attention Q,K,V,O params in billions
mlpB = np.array([0.40, 6.00])  # MLP gate/up/down params in billions
tot = embed + attn + mlpB  # totals: 0.46B and 8.0B
print('Tiny total ~', round(float(tot[0]), 2), 'B, MLP share', round(float(100 * mlpB[0] / tot[0]), 1), '%')  # 0.46B, 87%
print('8B total ~', round(float(tot[1]), 2), 'B, MLP share', round(float(100 * mlpB[1] / tot[1]), 1), '%')  # 8.0B, 75%
print('rule: params = embed + L*(attn Q,K,V,O + MLP + norms) + head')  # the accounting recipe
x = np.arange(len(names))  # bar positions 0 and 1
fig, ax = plt.subplots(figsize=(7, 4))  # make one stacked-bar panel
ax.bar(x, embed, label='embed+head')  # bottom chunk: lobby directory
ax.bar(x, attn, bottom=embed, label='attention')  # middle chunk: Q,K,V,O
ax.bar(x, mlpB, bottom=embed + attn, label='MLP (biggest)')  # top chunk: feedforward
ax.set_xticks(x)  # place ticks under the two bars
ax.set_xticklabels(names)  # label each tower by name
ax.set_ylabel('params (billions)')  # label the y axis
ax.set_title('Where the params live: MLP dominates, attention is small')  # panel title
ax.legend()  # show which color is which chunk
fig.tight_layout()  # tidy spacing around the panel
plt.show()  # render the figure


### §5 GQA — sharing sticky notes across heads

**In one sentence: grouped-query attention (GQA) keeps all 32 query heads but shares each key/value head across a group of 4, so kv_dim shrinks 4× and the KV-cache shrinks with it.**

The cache formula is `2 × L × kv_dim × seq × bytes`, and kv_dim = (H / query-heads) × kv-heads: with H = 4096, 32 query heads of width 128 each, and 8 kv-heads, kv_dim = 1024 instead of 4096. Watch it: the diagram shows 8 kv sticky-notes serving 32 query heads in groups of 4, and the prints prove full-head cache at 32K would be ~17.2GB while GQA pays only ~4.3GB — the same 4× factor as the head count.


In [ ]:
import numpy as np  # reuse numpy for the kv-dim arithmetic
import matplotlib.pyplot as plt  # reuse pyplot for the grouping diagram
np.random.seed(12)  # fix the seed (no randomness here, habit for parity)
H, nh, nkv = 4096, 32, 8  # width, query heads, kv heads (8B-class)
kv_dim = H // nh * nkv  # grouped kv width: 128 x 8 = 1024
full_dim = H  # no sharing: kv width = full 4096
print('kv width with GQA:', kv_dim, '(vs full', full_dim, ')')  # 1024 vs 4096: quarter the notes
L, seq = 32, 32768  # layers and 32K context for the price tags
full_gb = 2 * L * full_dim * seq * 2 / 1e9  # full-head cache GB at bf16
gqa_gb = 2 * L * kv_dim * seq * 2 / 1e9  # GQA cache GB at bf16
print('32K cache full heads:', round(full_gb, 1), 'GB vs GQA:', round(gqa_gb, 1), 'GB')  # ~17.2 vs ~4.3
fig, ax = plt.subplots(figsize=(8, 3.5))  # make one panel for the grouping diagram
ax.axis('off')  # hide axes: diagram, not data
ax.set_xlim(0, 10)  # fix placement range
ax.set_ylim(0, 4)  # fix placement range
for g in range(4):  # draw 4 of the 8 groups (space for labels)
    ax.text(1.5 + g * 1.1, 2.4, f'4 q\nheads', ha='center', va='center', fontsize=7, bbox=dict(boxstyle='round', facecolor='lightblue'))  # query group box
    ax.text(1.5 + g * 1.1, 1.1, '1 kv', ha='center', va='center', fontsize=7, bbox=dict(boxstyle='round', facecolor='orange'))  # shared kv box
    ax.annotate('', xy=(1.5 + g * 1.1, 1.5), xytext=(1.5 + g * 1.1, 2.0), arrowprops=dict(arrowstyle='->'))  # sharing arrow
ax.text(7.5, 1.8, '8 kv serve 32 q\n(cache / 4)', ha='center', va='center', fontsize=9, bbox=dict(boxstyle='round', facecolor='wheat'))  # moral box
ax.text(4, 3.4, 'GQA: groups of 4 query heads share one key/value head', ha='center', fontsize=9, fontweight='bold')  # banner
ax.set_title('Fewer sticky notes, same questions asked')  # diagram title
plt.tight_layout()  # snug the layout
plt.show()  # render the figure


### §6 KV-cache and context — the paper stack grows linearly

**In one sentence: the KV-cache holds 2 × L × kv_dim × seq × bytes (two sticky notes per token per layer), so doubling context doubles the cache — 4K → 32K is an 8× tax, 4K → 128K is 32×.**

The 8B-class curve makes it visceral: 0.5GB at 4K (invisible next to 16.1GB of weights), 4.3GB at 32K, 17.2GB at 128K (heavier than the model itself) — 16.1 + 17.2 > 24GB, so batch-1 128K no longer fits the card that swallowed 4K easily. Watch it: the growth curve against the flat red weights line, with the two fit verdicts printed (True at 4K, False at 128K).


In [ ]:
import numpy as np  # reuse numpy for cache math and curve points
import matplotlib.pyplot as plt  # reuse pyplot for the growth-curve panel
np.random.seed(12)  # fix the seed (habit: reproducible runs)
L, H, nh, nkv = 32, 4096, 32, 8  # 8B-class: 32 layers, width 4096, 8 kv heads
kv_dim = H // nh * nkv  # kv width per layer after GQA grouping (1024)
bpe = 2  # bytes per number in bf16
seqs = np.array([4096, 32768, 131072])  # context lengths: 4K, 32K, 128K
cache = 2 * L * kv_dim * seqs * bpe / 1e9  # KV-cache GB: 2(K+V)*L*kv_dim*seq*bytes
P = 8.03e9  # 8B-class param count from the lab table
weights = P * bpe / 1e9  # weight memory in GB at bf16
for s, c in zip(seqs, cache):  # loop over the three context lengths
    print(f'ctx {int(s) // 1024}K: cache {float(c):.1f} GB vs weights {float(weights):.1f} GB')  # the growing tax
print('fits 24GB card at 4K:', bool(weights + float(cache[0]) < 24))  # True: room left
print('fits 24GB card at 128K:', bool(weights + float(cache[2]) < 24))  # False: paper wins
fig, ax = plt.subplots(figsize=(7, 4))  # make one growth-curve panel
ax.plot(seqs / 1024, cache, marker='o')  # cache GB vs context in K
ax.axhline(weights, linestyle='--', color='red', label='weights 16.1 GB')  # flat weight line
ax.set_xlabel('context (K tokens)')  # label the x axis
ax.set_ylabel('KV-cache (GB, bf16, batch 1)')  # label the y axis
ax.set_title('Cache grows with context; weights stay flat')  # panel title
ax.legend()  # show which line is cache vs weights
fig.tight_layout()  # tidy spacing around the panel
plt.show()  # render the figure


### §7 Bytes and FLOPs — what a token really costs

**In one sentence: every number costs dtype bytes in memory (bf16 = 2, int8 = 1, int4 = 0.5 — halving the dtype halves weights and cache alike), and every generated token costs ~2P weight FLOPs plus an attention term 2×L×H×seq that grows with context.**

The bars price an 8B tower's weights (full-attn estimate, same recipe as the lab's flop math): 17.7GB in bf16, 8.8GB in int8, 4.4GB in int4 — quantization attacks the flat blue bar of the anchor. The attention term attacks the growing one: at 4K it adds ~1.1G to ~17.7G of weight work (6% share, total ≈ 18.7G), but at 32K it adds ~8.6G (~35% share, total ≈ 24.7G). Watch it: the dtype bars plus both printed flop budgets — same model, longer context, a third of each token's math spent re-reading the paper stack.


In [ ]:
import numpy as np  # reuse numpy for the byte and flop math
import matplotlib.pyplot as plt  # reuse pyplot for the dtype bars
np.random.seed(12)  # fix the seed (no randomness here, habit for parity)
P = 8.03e9  # 8B-class published params (matches §4 anchor 16.1GB bf16; full-attn recipe est ~8.84B differs only by head-counting, ignore here)
bpe = {'bf16': 2, 'int8': 1, 'int4': 0.5}  # bytes per number per dtype
for k, v in bpe.items():  # price the weights in each dtype
    print(f'8B weights {k}:', round(P * v / 1e9, 1), 'GB')  # expect 17.7 / 8.8 / 4.4
L, H = 32, 4096  # layers and width for the attention term
for seq in [4096, 32768]:  # short vs long context
    attn = 2 * L * H * seq  # scores + weighted-sum replaying the whole context
    total = 2 * P + attn  # weight term (flat) + attention term (grows)
    print(f'ctx {seq // 1024}K flop/tok:', round(total / 1e9, 1), 'G (attn share', round(100 * attn / total), '%)')  # 18.7G/6% and 26.3G/33%
fig, ax = plt.subplots(figsize=(6, 4))  # make one panel for the dtype bars
ax.bar(list(bpe), [P * v / 1e9 for v in bpe.values()], color=['steelblue', 'orange', 'green'])  # one bar per dtype
ax.set_ylabel('8B weights (GB)')  # label the memory axis
ax.set_title('Quantization shrinks the flat bar: 17.7 -> 8.8 -> 4.4 GB')  # title the takeaway
plt.tight_layout()  # snug the layout
plt.show()  # render the figure


### §8 Tied embeddings — one directory for entrance and exit

**In one sentence: tied embeddings share a single V×H matrix between the input token table and the output LM head, deleting one full matrix — for the 1B Llama (vocab 128,256 × width 2048) that saves ~262.7M params.**

Untied models pay the directory twice (once to read words in, once to score words out); tying says the two jobs can reuse the same fingerprints. The lab caught this live: its formula overshoots the published 1B total by ~0.26B — almost exactly one 128,256×2,048 matrix — because that model ties and the naive recipe counts both. Watch it: the diagram draws one shared table with two arrows (lookup in, projection out) beside the 262.7M receipt.


In [ ]:
import numpy as np  # reuse numpy for the savings arithmetic
import matplotlib.pyplot as plt  # reuse pyplot for the weight-reuse diagram
np.random.seed(12)  # fix the seed (no randomness here, habit for parity)
V, H = 128256, 2048  # 1B-Llama vocab and width
saved = V * H  # one shared-away matrix
print('tied-embedding saving:', saved, 'params')  # expect 262668288: one V*H matrix
print('in billions: ~', round(saved / 1e9, 2), 'B (the lab 1B overshoot explained)')  # ~0.26B
print('rule: tied models pay V*H once; untied pay it twice (embed + head)')  # the moral
fig, ax = plt.subplots(figsize=(8, 3.5))  # make one panel for the reuse diagram
ax.axis('off')  # hide axes: diagram, not data
ax.set_xlim(0, 10)  # fix placement range
ax.set_ylim(0, 4)  # fix placement range
ax.text(5, 2.0, 'ONE shared table\nV x H (262.7M)\nread + write', ha='center', va='center', fontsize=9, bbox=dict(boxstyle='round', facecolor='wheat'))  # shared box
ax.text(1.5, 3.0, 'tokens in\n(lookup)', ha='center', va='center', fontsize=9, bbox=dict(boxstyle='round', facecolor='lightblue'))  # entrance box
ax.text(1.5, 1.0, 'logits out\n(project)', ha='center', va='center', fontsize=9, bbox=dict(boxstyle='round', facecolor='lightgreen'))  # exit box
ax.text(8.5, 2.0, 'untied pays\nthis twice', ha='center', va='center', fontsize=9, bbox=dict(boxstyle='round', facecolor='tomato'))  # waste box
ax.annotate('', xy=(3.9, 2.2), xytext=(2.4, 2.8), arrowprops=dict(arrowstyle='->'))  # entrance to table
ax.annotate('', xy=(2.4, 1.2), xytext=(3.9, 1.8), arrowprops=dict(arrowstyle='->'))  # table to exit
ax.set_title('Tied embeddings: one matrix, two jobs (saves 262.7M on the 1B)')  # diagram title
plt.tight_layout()  # snug the layout
plt.show()  # render the figure


## How it all fits together — the system map (§9)

One config line walks the whole audit: **config** (L, H, heads, kv-heads, vocab) feeds the **param audit** (§4: 464,705 for Tiny, ~8B published for the tower), params × dtype set the **weights memory** (§7), layers × kv-heads × context × bytes set the **KV memory** (§5–§6), and weights + context set the **per-token cost** (§7) — with tying (§8) as the rebate. The trace below runs both ends: TinyGPT's entire world (weights ~0.9MB, cache 72KB at its 48-token window) beside the 8B tower's (16.1GB weights, 0.5 → 17.2GB cache) — same formulas, seven orders of magnitude apart.


In [ ]:
import numpy as np  # import numpy for the seed habit (diagram uses no randomness)
import matplotlib.pyplot as plt  # only pyplot draws: this cell is a pure diagram
np.random.seed(12)  # fix the seed for reproducibility
fig, ax = plt.subplots(figsize=(10, 3.5))  # wide figure for the left-to-right pipeline
ax.axis('off')  # hide axes: diagram, not data
ax.set_xlim(0, 14)  # fix placement range
ax.set_ylim(0, 4)  # fix placement range
stages = [('config §9\nL,H,heads,V', 1.2), ('audit §4\ncount all', 4), ('memory §5-§7\nweights+KV', 7), ('cost §7\nflop/tok', 10), ('fit?\nGPU?', 12.4)]  # pipeline stations
colors = ['wheat', 'lightblue', 'plum', 'lightgreen', 'steelblue']  # color per station type
for (name, xc), c in zip(stages, colors):  # draw each pipeline station
    ax.text(xc, 2.2, name, ha='center', va='center', fontsize=8,  # centered station label
            bbox=dict(boxstyle='round', facecolor=c))  # rounded colored box around it
ax.text(6.8, 3.5, 'AUDIT: config -> params (§4) -> bytes (§5-§7) -> flops (§7) -> GPU verdict', ha='center', fontsize=9, fontweight='bold')  # banner line
for xa, xb in [(2.1, 3.2), (4.8, 6.1), (7.9, 9.3), (10.7, 11.8)]:  # arrow gaps between stations
    ax.annotate('', xy=(xb, 2.2), xytext=(xa, 2.2), arrowprops=dict(arrowstyle='->'))  # forward arrow left -> right
ax.set_title('System map: one config line to a GPU verdict')  # diagram title
plt.tight_layout()  # snug the layout
plt.show()  # render the map


In [ ]:
import numpy as np  # reuse numpy for the tiny-vs-tower trace
np.random.seed(12)  # fix the seed (numbers below are hardcoded; habit kept)
tiny_P = 464705  # audited TinyGPT params (§4)
tiny_w = tiny_P * 2 / 1e6  # bf16 weights in MB
tiny_kv = 2 * 4 * 96 * 48 * 2  # Tiny KV bytes: 2 x L4 x kv96 x seq48 x bf16 (§6)
tower_P, tower_w = 8.03e9, 8.03e9 * 2 / 1e9  # published 8B params + bf16 GB (§4, §7)
tower_kv4k = 2 * 32 * 1024 * 4096 * 2 / 1e9  # 8B cache GB at 4K (§6)
tower_kv128k = 2 * 32 * 1024 * 131072 * 2 / 1e9  # 8B cache GB at 128K (§6)
print('TinyGPT: params', tiny_P, '-> weights', round(tiny_w, 2), 'MB + cache', round(tiny_kv / 1024, 1), 'KB')  # <1MB world
print('8B tower: params', round(tower_P / 1e9, 2), 'B -> weights', round(tower_w, 1), 'GB')  # 16.1GB cabinets
print('8B cache: 4K', round(tower_kv4k, 1), 'GB -> 128K', round(tower_kv128k, 1), 'GB (same formula, 32x tax)')  # paper explodes


## Bridge to the lab

Open `notebook.ipynb` now. Its cells run the same audit on real configs:

1. **Cell 1 (setup)** — seeds plus `os.makedirs('./results')`. Same `np.random.seed` habit as this lecture.
2. **Cell 2 (configs)** — TinyGPT-P11 plus Llama-3.2-1B, Llama-3.1-8B, Qwen2.5-7B, Gemma-2-9B (layers, hidden, heads, kv-heads, vocab, §4's audit inputs) — the towers your stacked bar summarized.
3. **Cell 3 (verify the accounting)** — the param formula checked exactly against the real P11 TinyGPT (464,705 by torch and by hand, match = True, §4's number), so you trust it on the big rows (8.03/8.03, 7.62/7.62 hits; the 1B "overshoot" of ~0.26B reveals tied embeddings, §8).
4. **Cell 4 (KV-cache + GPU fit)** — `kv_bytes()` computes your cache curve with GQA (§5–§6: 8B 0.5GB at 4K → 4.3GB at 32K → 17.2GB at 128K) and the weights verdict (24GB card fits 4K, not 128K batch > 1) into `results/kv_cache.csv` and `kv_cache.png`.
5. **Cell 5 (flops + scaling proof)** — decode flops/token ≈ 18.7G at 4K → ≈ 26.3G at 32K (attention share 6% → 33%, §7) plus a measured timing slope ≈ 1.76 ≈ quadratic — the paper-stack tax made visible, motivating Session 6 quantization (bf16 → int8 → int4 on this same table).


## Self-check — predict first, then check

> Retrieval rule: cover every *Answer* below, write your prediction (a number, a shape, or one sentence), run the § cell it cites, then uncover. The last question in each set has no printed answer — do it from the map.



1. **An 8B bf16 model at 4K context: roughly how much memory for weights vs KV-cache?**

*Answer: weights ~16.1GB (8.03B × 2 bytes), cache only ~0.5GB (§6) — weights dominate at short context, which is why the anchor's blue bar towers over the green one.*

2. **What happens to the cache at 128K, and does it still fit a 24GB card?**

*Answer: it grows 32× to ~17.2GB, exceeding the weights (§6) — 16.1 + 17.2 > 24GB, so batch-1 128K no longer fits. Rerun the §6 cell and read the False verdict.*

3. **Why do GQA (fewer kv-heads) and int8 cache each roughly halve the cache — and what does tying save?**

*Answer: cache = 2 × L × kv_dim × seq × bytes, so halving kv-heads halves kv_dim (§5: 8 kv-heads quarter it vs full) and int8 halves the bytes factor (§7) — while tying deletes one V×H matrix outright (§8: 262.7M on the 1B Llama).*
